<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/Plitidepsin_similar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 41.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from google.colab import data_table

# Maintain clear interactive tracking charts in Colab
data_table.enable_dataframe_formatter()

def run_production_analog_deduplication(sdf_input="Plitidepsin_Similar.sdf",
                                         csv_input="Plitidepsin_Similar.csv"):
    """
    Ingests the newly renamed PubChem analog files, strips salt counterions,
    calculates stereospecific InChIKeys, resolves redundancies, and preserves
    intentional stereoisomeric variants.
    """
    print(f"🧼 Initializing analog deduplication routine on target file: '{sdf_input}'...")

    # Initialize the RDKit Salt Remover to clean up counterions (TFA, HCl, etc.)
    remover = SaltRemover()

    # Load the molecules from your uploaded master SDF
    try:
        supplier = Chem.SDMolSupplier(sdf_input, sanitize=True, removeHs=True)
    except Exception as e:
        print(f"❌ Failed to parse '{sdf_input}'. Make sure it is completely uploaded to the sidebar folder.")
        return None

    # Load the accompanying PubChem metadata sheet if needed for cross-referencing
    try:
        df_metadata = pd.read_csv(csv_input)
        print(f"📊 Connected to metadata sheet containing {len(df_metadata)} data rows.\n")
    except Exception:
        print(f"⚠️ Metadata sheet '{csv_input}' not found. Defaulting to SDF properties only.\n")

    seen_inchikeys = set()
    cleaned_records = []

    duplicate_count = 0
    salt_stripped_count = 0

    for i, mol in enumerate(supplier):
        if mol is None:
            continue

        # Extract the PubChem Compound ID (CID) or default name
        cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Analog_Index_{i}"

        # 1. Execute Salt/Counterion Stripping to yield the clean parent molecule
        stripped_mol = remover.StripMol(mol)

        # Track if the molecule lost weight/atoms during stripping
        if stripped_mol.GetNumAtoms() < mol.GetNumAtoms():
            salt_stripped_count += 1

        # 2. Compute the stereospecific InChIKey
        try:
            inchikey = Chem.MolToInchiKey(stripped_mol)
        except AttributeError:
            # Handle RDKit variation formatting styles
            inchikey = Chem.InchiToInchiKey(Chem.MolToInchi(stripped_mol))
        except Exception:
            # Fallback to isomeric SMILES string representation if needed
            inchikey = Chem.MolToSmiles(stripped_mol, isomericSmiles=True)

        # 3. Filter Duplicates based on the full structural string signature
        if inchikey in seen_inchikeys:
            duplicate_count += 1
            continue

        seen_inchikeys.add(inchikey)

        # Convert back to clean canonical isomeric SMILES representation
        canonical_smiles = Chem.MolToSmiles(stripped_mol, isomericSmiles=True)

        cleaned_records.append({
            'Compound_ID': f"CID_{cid}" if not str(cid).startswith("CID") else cid,
            'Analog_SMILES': canonical_smiles,
            'InChIKey_Hash': inchikey,
            'Parent_Scaffold_Source': 'Plitidepsin 97% Analog Cluster'
        })

    # Compile the pristine data matrix
    df_clean_analogs = pd.DataFrame(cleaned_records)

    # Save out the definitive deduplicated 2D master base file
    output_csv = "Plitidepsin_Analogs_Clean_Base_2D.csv"
    df_clean_analogs.to_csv(output_csv, index=False)

    print("✨ Analog Deduplication and Salt Stripping Subroutine Complete!")
    print(f"📊 Total raw entries processed from SDF: {len(supplier)}")
    print(f"✂️ Redundant duplicate records discarded: {duplicate_count}")
    print(f"Salt molecules successfully stripped: {salt_stripped_count}")
    print(f"🎯 Total unique analog leads preserved: {len(df_clean_analogs)}")
    print(f"📁 Deduplicated collection exported to: '{output_csv}'")

    return df_clean_analogs

# --- Execute Deduplication Framework ---
df_clean_analogs = run_production_analog_deduplication()

# View the clean interactive database matrix
df_clean_analogs

🧼 Initializing analog deduplication routine on target file: 'Plitidepsin_Similar.sdf'...
📊 Connected to metadata sheet containing 570 data rows.

✨ Analog Deduplication and Salt Stripping Subroutine Complete!
📊 Total raw entries processed from SDF: 570
✂️ Redundant duplicate records discarded: 4
Salt molecules successfully stripped: 4
🎯 Total unique analog leads preserved: 566
📁 Deduplicated collection exported to: 'Plitidepsin_Analogs_Clean_Base_2D.csv'


,Compound_ID,Analog_SMILES,InChIKey_Hash,Parent_Scaffold_Source
0,CID_122651,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-SJPGYWQQSA-N,Plitidepsin 97% Analog Cluster
1,CID_9812534,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,UUSZLLQJYRSZIS-LXNNNBEUSA-N,Plitidepsin 97% Analog Cluster
2,CID_123844,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,XQZOGOCTPKFYKC-VSZULPIASA-N,Plitidepsin 97% Analog Cluster
3,CID_13895197,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-XYUQHQMCSA-N,Plitidepsin 97% Analog Cluster
4,CID_44298771,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-OXHMIFALSA-N,Plitidepsin 97% Analog Cluster
...,...,...,...,...
561,CID_177575987,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,BHILFKFEJCTGDH-DEOZJFGJSA-N,Plitidepsin 97% Analog Cluster
562,CID_177579848,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,KYHUYMLIVQFXRI-GKHUWPTNSA-N,Plitidepsin 97% Analog Cluster
563,CID_177594766,CCC(=O)N(C)[C@H](CC(C)C)C(=O)NC1C(=O)NC([C@@H]...,DLFRRLJDRQYRGC-GKCFDRQGSA-N,Plitidepsin 97% Analog Cluster
564,CID_177596293,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...,APOIETXEKDWFTR-CPQQIORRSA-N,Plitidepsin 97% Analog Cluster


In [ ]:
import pandas as pd
from rdkit import Chem
from google.colab import data_table

# Maintain interactive spreadsheet rendering
data_table.enable_dataframe_formatter()

def run_production_analog_sanitization(input_csv="Plitidepsin_Analogs_Clean_Base_2D.csv"):
    """
    Ingests the deduplicated 2D analog library, executes strict electronic
    sanitization/valence auditing, and creates a step-by-step audit record.
    """
    print(f"🧬 Loading unique analog database: '{input_csv}'...")
    try:
        df_base = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify Step 1 finished completely.")
        return None

    audit_rows = []
    failed_structures_count = 0
    passed_structures_count = 0

    print(f"⚡ Processing {len(df_base)} compounds through structural stabilization...\n")

    for idx, row in df_base.iterrows():
        comp_id = row['Compound_ID']
        raw_smiles = row['Analog_SMILES']
        base_inchikey = row['InChIKey_Hash']

        # Parse molecule from the stored unique SMILES string
        mol = Chem.MolFromSmiles(raw_smiles)

        # --- Entry Stage 1: Record Deduplicated State ---
        audit_rows.append({
            'Compound_ID': comp_id,
            'Pipeline_Stage': '1_Deduplicated',
            'Resulting_SMILES': raw_smiles,
            'InChIKey_Hash': base_inchikey,
            'Valence_Status': 'Awaiting Audit'
        })

        # --- Entry Stage 2: Clean, Sanitize, and Audit Valence ---
        if mol is not None:
            try:
                # Force RDKit to resolve aromaticity rings, radical states, and hypervalency
                Chem.SanitizeMol(mol)
                Chem.Cleanup(mol)

                # Re-generate standard canonical configuration after electronic cleanup
                sanitized_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                sanitized_inchikey = Chem.MolToInchiKey(mol)
                valence_status = 'PASS (Valid Valence)'
                passed_structures_count += 1

            except Exception as valence_error:
                sanitized_smiles = "STSTRUCTURE_CRASH"
                sanitized_inchikey = base_inchikey
                valence_status = f"FAIL (Valence Error: {str(valence_error)})"
                failed_structures_count += 1
        else:
            sanitized_smiles = "INVALID_SMILES_FORMAT"
            sanitized_inchikey = base_inchikey
            valence_status = "FAIL (Parsing Error)"
            failed_structures_count += 1

        audit_rows.append({
            'Compound_ID': comp_id,
            'Pipeline_Stage': '2_Sanitized',
            'Resulting_SMILES': sanitized_smiles,
            'InChIKey_Hash': sanitized_inchikey,
            'Valence_Status': valence_status
        })

    # Compile the final step-by-step comparison dataset
    df_sanitization_audit = pd.DataFrame(audit_rows)

    # Save a permanent audit sheet to the disk workspace
    df_sanitization_audit.to_csv("Plitidepsin_Analogs_Sanitization_Audit.csv", index=False)

    # Generate a clean, filtered master file containing only valid structures for the next step
    df_production_ready = df_sanitization_audit[
        (df_sanitization_audit['Pipeline_Stage'] == '2_Sanitized') &
        (df_sanitization_audit['Valence_Status'] == 'PASS (Valid Valence)')
    ].copy()

    # Keep only primary operational tracking columns
    df_production_ready = df_production_ready.drop(columns=['Pipeline_Stage'])
    df_production_ready.to_csv("Plitidepsin_Analogs_Sanitized_Master_2D.csv", index=False)

    print("✨ High-Throughput Sanitization & Valence Audit Complete!")
    print(f"🟩 Structurally verified chemical structures: {passed_structures_count}")
    print(f"🟥 Structurally impossible compounds discarded: {failed_structures_count}")
    print(f"📁 Step-by-step history log saved as: 'Plitidepsin_Analogs_Sanitization_Audit.csv'")
    print(f"📁 Clean master library ready for Step 6 saved as: 'Plitidepsin_Analogs_Sanitized_Master_2D.csv'")

    return df_sanitization_audit

# --- Run the Electronic Sanitization Audit ---
df_sanitization_audit = run_production_analog_sanitization()

# Display the interactive dual-row comparison matrix
df_sanitization_audit

🧬 Loading unique analog database: 'Plitidepsin_Analogs_Clean_Base_2D.csv'...
⚡ Processing 566 compounds through structural stabilization...

✨ High-Throughput Sanitization & Valence Audit Complete!
🟩 Structurally verified chemical structures: 566
🟥 Structurally impossible compounds discarded: 0
📁 Step-by-step history log saved as: 'Plitidepsin_Analogs_Sanitization_Audit.csv'
📁 Clean master library ready for Step 6 saved as: 'Plitidepsin_Analogs_Sanitized_Master_2D.csv'


,Compound_ID,Pipeline_Stage,Resulting_SMILES,InChIKey_Hash,Valence_Status
0,CID_122651,1_Deduplicated,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-SJPGYWQQSA-N,Awaiting Audit
1,CID_122651,2_Sanitized,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-SJPGYWQQSA-N,PASS (Valid Valence)
2,CID_9812534,1_Deduplicated,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,UUSZLLQJYRSZIS-LXNNNBEUSA-N,Awaiting Audit
3,CID_9812534,2_Sanitized,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,UUSZLLQJYRSZIS-LXNNNBEUSA-N,PASS (Valid Valence)
4,CID_123844,1_Deduplicated,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,XQZOGOCTPKFYKC-VSZULPIASA-N,Awaiting Audit
...,...,...,...,...,...
1127,CID_177594766,2_Sanitized,CCC(=O)N(C)[C@H](CC(C)C)C(=O)NC1C(=O)NC([C@@H]...,DLFRRLJDRQYRGC-GKCFDRQGSA-N,PASS (Valid Valence)
1128,CID_177596293,1_Deduplicated,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...,APOIETXEKDWFTR-CPQQIORRSA-N,Awaiting Audit
1129,CID_177596293,2_Sanitized,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...,APOIETXEKDWFTR-CPQQIORRSA-N,PASS (Valid Valence)
1130,CID_177602217,1_Deduplicated,CCCCCCCCCCCCCCCCCC(=O)N(C)[C@H](CC(C)C)C(=O)N[...,BLTXIACVGKWQDB-FQDGNKKHSA-N,Awaiting Audit


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from google.colab import data_table

# Enable interactive table display for your step verification
data_table.enable_dataframe_formatter()

def run_production_desalting_pipeline(input_csv="Plitidepsin_Analogs_Sanitized_Master_2D.csv"):
    """
    Ingests the sanitized analog library, applies RDKit's SaltRemover,
    and extracts the largest molecular fragment to isolate pure parent frames.
    """
    print(f"🧬 Loading sanitized dataset: '{input_csv}'...")
    try:
        df_sanitized = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Ensure the previous step completed successfully.")
        return None

    # Initialize the RDKit SaltRemover
    remover = SaltRemover()

    desalting_records = []
    fragments_isolated_count = 0

    print(f"🧂 Commencing multi-fragment stripping on {len(df_sanitized)} compounds...\n")

    for idx, row in df_sanitized.iterrows():
        comp_id = row['Compound_ID']
        sanitized_smiles = row['Resulting_SMILES']
        inchikey = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(sanitized_smiles)
        if mol is None:
            continue

        # 1. Strip standard counter-ions/salts
        cleansed_mol = remover.StripMol(mol, dontRemoveEverything=True)

        # 2. Extract and evaluate molecular fragments
        frags = Chem.GetMolFrags(cleansed_mol, asMols=True)

        if frags:
            # Isolate the largest fragment based on total atom count
            largest_frag = max(frags, default=cleansed_mol, key=lambda m: m.GetNumAtoms())
            desalted_smiles = Chem.MolToSmiles(largest_frag, isomericSmiles=True)

            # Check if multi-fragment mixtures were successfully separated
            if len(frags) > 1:
                fragments_isolated_count += 1
        else:
            desalted_smiles = Chem.MolToSmiles(cleansed_mol, isomericSmiles=True)

        # Calculate updated InChIKey for the pristine isolated parent frame
        desalted_inchikey = Chem.MolToInchiKey(Chem.MolFromSmiles(desalted_smiles))

        desalting_records.append({
            'Compound_ID': comp_id,
            'Sanitized_SMILES': sanitized_smiles,
            'Desalted_SMILES': desalted_smiles,
            'Original_InChIKey': inchikey,
            'Desalted_InChIKey': desalted_inchikey
        })

    # Compile the final data comparison grid
    df_desalted_report = pd.DataFrame(desalting_records)

    # Save a permanent copy of the desalting history log to workspace disk
    df_desalted_report.to_csv("Plitidepsin_Analogs_Desalting_History_Log.csv", index=False)

    # Export the clean, optimized 2D master file for your next step
    df_next_step_input = df_desalted_report[['Compound_ID', 'Desalted_SMILES', 'Desalted_InChIKey']].copy()
    df_next_step_input.columns = ['Compound_ID', 'SMILES', 'InChIKey_Hash']
    df_next_step_input.to_csv("Plitidepsin_Analogs_Desalted_Master_2D.csv", index=False)

    print("✨ High-Throughput Desalting & Fragment Isolation Complete!")
    print(f"🟩 Total compounds successfully verified and processed: {len(df_desalted_report)}")
    print(f"✂️ Multi-fragment mixtures separated: {fragments_isolated_count}")
    print(f"📁 Detailed desalting log saved as: 'Plitidepsin_Analogs_Desalting_History_Log.csv'")
    print(f"📁 Isolated parent library ready for next stage saved as: 'Plitidepsin_Analogs_Desalted_Master_2D.csv'")

    return df_desalted_report

# --- Run the Desalting Filter Loop ---
df_desalted_report = run_production_desalting_pipeline()

# Display the interactive comparison matrix
df_desalted_report

🧬 Loading sanitized dataset: 'Plitidepsin_Analogs_Sanitized_Master_2D.csv'...
🧂 Commencing multi-fragment stripping on 566 compounds...

✨ High-Throughput Desalting & Fragment Isolation Complete!
🟩 Total compounds successfully verified and processed: 566
✂️ Multi-fragment mixtures separated: 2
📁 Detailed desalting log saved as: 'Plitidepsin_Analogs_Desalting_History_Log.csv'
📁 Isolated parent library ready for next stage saved as: 'Plitidepsin_Analogs_Desalted_Master_2D.csv'


,Compound_ID,Sanitized_SMILES,Desalted_SMILES,Original_InChIKey,Desalted_InChIKey
0,CID_122651,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-SJPGYWQQSA-N,KYHUYMLIVQFXRI-SJPGYWQQSA-N
1,CID_9812534,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,UUSZLLQJYRSZIS-LXNNNBEUSA-N,UUSZLLQJYRSZIS-LXNNNBEUSA-N
2,CID_123844,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,XQZOGOCTPKFYKC-VSZULPIASA-N,XQZOGOCTPKFYKC-VSZULPIASA-N
3,CID_13895197,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-XYUQHQMCSA-N,KYHUYMLIVQFXRI-XYUQHQMCSA-N
4,CID_44298771,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-OXHMIFALSA-N,KYHUYMLIVQFXRI-OXHMIFALSA-N
...,...,...,...,...,...
561,CID_177575987,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,BHILFKFEJCTGDH-DEOZJFGJSA-N,BHILFKFEJCTGDH-DEOZJFGJSA-N
562,CID_177579848,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,KYHUYMLIVQFXRI-GKHUWPTNSA-N,KYHUYMLIVQFXRI-GKHUWPTNSA-N
563,CID_177594766,CCC(=O)N(C)[C@H](CC(C)C)C(=O)NC1C(=O)NC([C@@H]...,CCC(=O)N(C)[C@H](CC(C)C)C(=O)NC1C(=O)NC([C@@H]...,DLFRRLJDRQYRGC-GKCFDRQGSA-N,DLFRRLJDRQYRGC-GKCFDRQGSA-N
564,CID_177596293,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...,APOIETXEKDWFTR-CPQQIORRSA-N,APOIETXEKDWFTR-CPQQIORRSA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from google.colab import data_table

# Keep table formatting fully interactive in Colab
data_table.enable_dataframe_formatter()

def run_production_standardization_pipeline(input_csv="Plitidepsin_Analogs_Desalted_Master_2D.csv"):
    """
    Ingests the desalted 2D parent library, standardizes charge variations,
    and normalizes polarized functional group definitions to prevent 3D embedding failures.
    """
    print(f"🧬 Loading desalted master library: '{input_csv}'...")
    try:
        df_desalted = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Confirm your desalting step completed successfully.")
        return None

    standardized_records = []
    modifications_count = 0

    print(f"⚡ Normalizing electronic structures for {len(df_desalted)} analogs...\n")

    for idx, row in df_desalted.iterrows():
        comp_id = row['Compound_ID']
        desalted_smiles = row['SMILES']
        old_inchikey = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(desalted_smiles)
        if mol is None:
            continue

        try:
            # 1. Initialize RDKit's chemical structure standardizer
            standardized_mol = rdMolStandardize.StandardizeMol(mol)
            standardized_smiles = Chem.MolToSmiles(standardized_mol, isomericSmiles=True)
            standardized_inchikey = Chem.MolToInchiKey(standardized_mol)

            # 2. Track if any charge distributions or valence drawings were modified
            if standardized_smiles != desalted_smiles:
                modifications_count += 1

        except Exception:
            # Safe fallback if an exotic resonance pattern hits an exception
            standardized_smiles = desalted_smiles
            standardized_inchikey = old_inchikey

        standardized_records.append({
            'Compound_ID': comp_id,
            'Desalted_SMILES': desalted_smiles,
            'Standardized_SMILES': standardized_smiles,
            'Original_InChIKey': old_inchikey,
            'Standardized_InChIKey': standardized_inchikey
        })

    # Compile tracking dataframe
    df_std_report = pd.DataFrame(standardized_records)

    # Save a permanent copy of the standardization history to disk
    df_std_report.to_csv("Plitidepsin_Analogs_Standardization_History.csv", index=False)

    # Export clean 2D dataset ready for the upcoming tautomer tracking stage
    df_next_stage = df_std_report[['Compound_ID', 'Standardized_SMILES', 'Standardized_InChIKey']].copy()
    df_next_stage.columns = ['Compound_ID', 'SMILES', 'InChIKey_Hash']
    df_next_stage.to_csv("Plitidepsin_Analogs_Standardized_Master_2D.csv", index=False)

    print("✨ Functional Group Standardization Complete!")
    print(f"🟩 Total compounds successfully standardized and verified: {len(df_std_report)}")
    print(f"⚖️ Resonance frameworks/charges normalized: {modifications_count}")
    print(f"📁 Detailed transformation log saved as: 'Plitidepsin_Analogs_Standardization_History.csv'")
    print(f"📁 Standardized collection ready for next phase saved as: 'Plitidepsin_Analogs_Standardized_Master_2D.csv'")

    return df_std_report

# --- Execute Standardization Loop ---
df_std_report = run_production_standardization_pipeline()

# Display interactive verification matrix
df_std_report

🧬 Loading desalted master library: 'Plitidepsin_Analogs_Desalted_Master_2D.csv'...
⚡ Normalizing electronic structures for 566 analogs...

✨ Functional Group Standardization Complete!
🟩 Total compounds successfully standardized and verified: 566
⚖️ Resonance frameworks/charges normalized: 0
📁 Detailed transformation log saved as: 'Plitidepsin_Analogs_Standardization_History.csv'
📁 Standardized collection ready for next phase saved as: 'Plitidepsin_Analogs_Standardized_Master_2D.csv'


,Compound_ID,Desalted_SMILES,Standardized_SMILES,Original_InChIKey,Standardized_InChIKey
0,CID_122651,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-SJPGYWQQSA-N,KYHUYMLIVQFXRI-SJPGYWQQSA-N
1,CID_9812534,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,UUSZLLQJYRSZIS-LXNNNBEUSA-N,UUSZLLQJYRSZIS-LXNNNBEUSA-N
2,CID_123844,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,XQZOGOCTPKFYKC-VSZULPIASA-N,XQZOGOCTPKFYKC-VSZULPIASA-N
3,CID_13895197,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-XYUQHQMCSA-N,KYHUYMLIVQFXRI-XYUQHQMCSA-N
4,CID_44298771,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,KYHUYMLIVQFXRI-OXHMIFALSA-N,KYHUYMLIVQFXRI-OXHMIFALSA-N
...,...,...,...,...,...
561,CID_177575987,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,BHILFKFEJCTGDH-DEOZJFGJSA-N,BHILFKFEJCTGDH-DEOZJFGJSA-N
562,CID_177579848,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...,KYHUYMLIVQFXRI-GKHUWPTNSA-N,KYHUYMLIVQFXRI-GKHUWPTNSA-N
563,CID_177594766,CCC(=O)N(C)[C@H](CC(C)C)C(=O)NC1C(=O)NC([C@@H]...,CCC(=O)N(C)[C@H](CC(C)C)C(=O)NC1C(=O)NC([C@@H]...,DLFRRLJDRQYRGC-GKCFDRQGSA-N,DLFRRLJDRQYRGC-GKCFDRQGSA-N
564,CID_177596293,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...,APOIETXEKDWFTR-CPQQIORRSA-N,APOIETXEKDWFTR-CPQQIORRSA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from google.colab import data_table

# Maintain clear interactive tracking charts in Colab
data_table.enable_dataframe_formatter()

def run_production_drug_likeness(input_csv="Plitidepsin_Analogs_Standardization_Master_2D.csv"):
    """
    Ingests the standardized 2D analog dataset, calculates core Lipinski & Veber
    descriptors, logs standard violations, and applies macrocyclic space filters.
    """
    print(f"🧬 Loading standardized analog library: '{input_csv}'...")
    try:
        df_master = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify the standardization cell completed cleanly.")
        return None

    profiling_records = []
    passed_filter_count = 0
    failed_filter_count = 0

    print(f"📊 Running descriptor math across {len(df_master)} prioritized scaffolds...\n")

    for idx, row in df_master.iterrows():
        comp_id = row['Compound_ID']
        smiles = row['SMILES']
        inchikey = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue

        # 1. High-Precision Property Extractions
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Lipinski.NumHDonors(mol)
        hba = Lipinski.NumHAcceptors(mol)
        tpsa = Descriptors.TPSA(mol)
        rotb = Lipinski.NumRotatableBonds(mol)

        # 2. Traditional Violations Counter (Your exact logic)
        lipinski_violations = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
        veber_violations = sum([tpsa > 140, rotb > 10])

        # 3. Targeted Macrocyclic (bRo5) Filter Application
        if mw <= 1500 and logp <= 9.0 and tpsa <= 350:
            filter_status = "PASS (bRo5 Compliant)"
            passed_filter_count += 1
        else:
            filter_status = "FAIL (Physicochemical Outlier)"
            failed_filter_count += 1

        profiling_records.append({
            'Compound_ID': comp_id,
            'MW_g_mol': round(mw, 2),
            'LogP': round(logp, 2),
            'HBD': hbd,
            'HBA': hba,
            'TPSA_Å²': round(tpsa, 2),
            'RotB': rotb,
            'Lipinski_Violations': lipinski_violations,
            'Veber_Violations': veber_violations,
            'Filter_Status': filter_status,
            'InChIKey_Hash': inchikey,
            'SMILES': smiles
        })

    # Generate output dataframes
    df_drug_likeness = pd.DataFrame(profiling_records)

    # Save a permanent copy of the descriptor dataset to disk workspace
    df_drug_likeness.to_csv("Plitidepsin_Analogs_Drug_Likeness_Profiles.csv", index=False)

    # Export a cleaned master subset containing only the passed leads for Step 7
    df_passed_leads = df_drug_likeness[df_drug_likeness['Filter_Status'] == "PASS (bRo5 Compliant)"].copy()
    df_passed_leads = df_passed_leads[['Compound_ID', 'SMILES', 'InChIKey_Hash']]
    df_passed_leads.to_csv("Plitidepsin_Analogs_Profiled_Master_2D.csv", index=False)

    print("✨ High-Throughput Drug-Likeness Profiling Complete!")
    print(f"🟩 Analogs within target macrocyclic space: {passed_filter_count}")
    print(f"🟥 Extreme structural outliers rejected: {failed_filter_count}")
    print(f"📁 Detailed descriptors report saved as: 'Plitidepsin_Analogs_Drug_Likeness_Profiles.csv'")
    print(f"📁 Screened master collection ready for next phase saved as: 'Plitidepsin_Analogs_Profiled_Master_2D.csv'")

    return df_drug_likeness

# --- Run Descriptor Pipeline ---
df_drug_likeness = run_production_drug_likeness()

# View interactive properties chart
df_drug_likeness

🧬 Loading standardized analog library: 'Plitidepsin_Analogs_Standardization_Master_2D.csv'...
❌ Error: Could not locate 'Plitidepsin_Analogs_Standardization_Master_2D.csv'. Verify the standardization cell completed cleanly.


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from google.colab import data_table

# Maintain clear interactive tracking charts in Colab
data_table.enable_dataframe_formatter()

# CHANGED: 'Standardization' to 'Standardized' to match your sidebar file exactly
def run_production_drug_likeness(input_csv="Plitidepsin_Analogs_Standardized_Master_2D.csv"):
    """
    Ingests the standardized 2D analog dataset, calculates core Lipinski & Veber
    descriptors, logs standard violations, and applies macrocyclic space filters.
    """
    print(f"🧬 Loading standardized analog library: '{input_csv}'...")
    try:
        df_master = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify the standardization cell completed cleanly.")
        return None

    profiling_records = []
    passed_filter_count = 0
    failed_filter_count = 0

    print(f"📊 Running descriptor math across {len(df_master)} prioritized scaffolds...\n")

    for idx, row in df_master.iterrows():
        comp_id = row['Compound_ID']
        smiles = row['SMILES']
        inchikey = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue

        # 1. High-Precision Property Extractions
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Lipinski.NumHDonors(mol)
        hba = Lipinski.NumHAcceptors(mol)
        tpsa = Descriptors.TPSA(mol)
        rotb = Lipinski.NumRotatableBonds(mol)

        # 2. Traditional Violations Counter
        lipinski_violations = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
        veber_violations = sum([tpsa > 140, rotb > 10])

        # 3. Targeted Macrocyclic (bRo5) Filter Application
        if mw <= 1000 and logp <= 9.0 and tpsa <= 350:
            filter_status = "PASS (bRo5 Compliant)"
            passed_filter_count += 1
        else:
            filter_status = "FAIL (Physicochemical Outlier)"
            failed_filter_count += 1

        profiling_records.append({
            'Compound_ID': comp_id,
            'MW_g_mol': round(mw, 2),
            'LogP': round(logp, 2),
            'HBD': hbd,
            'HBA': hba,
            'TPSA_Å²': round(tpsa, 2),
            'RotB': rotb,
            'Lipinski_Violations': lipinski_violations,
            'Veber_Violations': veber_violations,
            'Filter_Status': filter_status,
            'InChIKey_Hash': inchikey,
            'SMILES': smiles
        })

    # Generate output dataframes
    df_drug_likeness = pd.DataFrame(profiling_records)

    # Save a permanent copy of the descriptor dataset to disk workspace
    df_drug_likeness.to_csv("Plitidepsin_Analogs_Drug_Likeness_Profiles.csv", index=False)

    # Export a cleaned master subset containing only the passed leads for Step 7
    df_passed_leads = df_drug_likeness[df_drug_likeness['Filter_Status'] == "PASS (bRo5 Compliant)"].copy()
    df_passed_leads = df_passed_leads[['Compound_ID', 'SMILES', 'InChIKey_Hash']]
    df_passed_leads.to_csv("Plitidepsin_Analogs_Profiled_Master_2D.csv", index=False)

    print("✨ High-Throughput Drug-Likeness Profiling Complete!")
    print(f"🟩 Analogs within target macrocyclic space: {passed_filter_count}")
    print(f"🟥 Extreme structural outliers rejected: {failed_filter_count}")
    print(f"📁 Detailed descriptors report saved as: 'Plitidepsin_Analogs_Drug_Likeness_Profiles.csv'")
    print(f"📁 Screened master collection ready for next phase saved as: 'Plitidepsin_Analogs_Profiled_Master_2D.csv'")

    return df_drug_likeness

# --- Run Descriptor Pipeline ---
df_drug_likeness = run_production_drug_likeness()

# View interactive properties chart
df_drug_likeness

🧬 Loading standardized analog library: 'Plitidepsin_Analogs_Standardized_Master_2D.csv'...
📊 Running descriptor math across 566 prioritized scaffolds...

✨ High-Throughput Drug-Likeness Profiling Complete!
🟩 Analogs within target macrocyclic space: 108
🟥 Extreme structural outliers rejected: 458
📁 Detailed descriptors report saved as: 'Plitidepsin_Analogs_Drug_Likeness_Profiles.csv'
📁 Screened master collection ready for next phase saved as: 'Plitidepsin_Analogs_Profiled_Master_2D.csv'


,Compound_ID,MW_g_mol,LogP,HBD,HBA,TPSA_Å²,RotB,Lipinski_Violations,Veber_Violations,Filter_Status,InChIKey_Hash,SMILES
0,CID_122651,1112.37,2.32,5,15,287.90,15,2,2,FAIL (Physicochemical Outlier),KYHUYMLIVQFXRI-SJPGYWQQSA-N,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...
1,CID_9812534,1110.36,2.52,4,15,284.74,15,2,2,FAIL (Physicochemical Outlier),UUSZLLQJYRSZIS-LXNNNBEUSA-N,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...
2,CID_123844,943.19,2.71,5,13,239.08,13,2,2,PASS (bRo5 Compliant),XQZOGOCTPKFYKC-VSZULPIASA-N,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...
3,CID_13895197,1112.37,2.32,5,15,287.90,15,2,2,FAIL (Physicochemical Outlier),KYHUYMLIVQFXRI-XYUQHQMCSA-N,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...
4,CID_44298771,1112.37,2.32,5,15,287.90,15,2,2,FAIL (Physicochemical Outlier),KYHUYMLIVQFXRI-OXHMIFALSA-N,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...
...,...,...,...,...,...,...,...,...,...,...,...,...
561,CID_177575987,1038.29,2.47,5,14,259.39,14,2,2,FAIL (Physicochemical Outlier),BHILFKFEJCTGDH-DEOZJFGJSA-N,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...
562,CID_177579848,1112.37,2.32,5,15,287.90,15,2,2,FAIL (Physicochemical Outlier),KYHUYMLIVQFXRI-GKHUWPTNSA-N,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C(...
563,CID_177594766,999.26,3.35,4,13,247.36,14,2,2,PASS (bRo5 Compliant),DLFRRLJDRQYRGC-GKCFDRQGSA-N,CCC(=O)N(C)[C@H](CC(C)C)C(=O)NC1C(=O)NC([C@@H]...
564,CID_177596293,1098.39,2.79,5,15,270.83,16,2,2,FAIL (Physicochemical Outlier),APOIETXEKDWFTR-CPQQIORRSA-N,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C[...


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from google.colab import data_table

# Maintain interactive spreadsheet layout in Colab
data_table.enable_dataframe_formatter()

def run_production_physiological_tautomerization(input_csv="Plitidepsin_Analogs_Profiled_Master_2D.csv"):
    """
    Ingests the 108 screened macrocycles, re-equilibrates localized proton charges,
    and locks down the definitive physiological canonical tautomer for 3D coordinate mapping.
    """
    print(f"🧬 Loading screened macrocyclic library: '{input_csv}'...")
    try:
        df_profiled = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify the descriptor filter cell finished cleanly.")
        return None

    # Initialize RDKit's uncharging engine and tautomer canonicalizer
    uncharger = rdMolStandardize.Uncharger()
    tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

    tautomer_records = []
    tautomers_shifted_count = 0

    print(f"⚡ Processing {len(df_profiled)} molecules through physiological equilibrium loops...\n")

    for idx, row in df_profiled.iterrows():
        comp_id = row['Compound_ID']
        profiled_smiles = row['SMILES']
        old_inchikey = row['InChIKey_Hash']

        mol = Chem.MolFromSmiles(profiled_smiles)
        if mol is None:
            continue

        try:
            # 1. Neutralize transient charges to balance proton states
            mol_uncharged = uncharger.uncharge(mol)

            # 2. Determine the most thermodynamically stable canonical tautomer form
            canonical_mol = tautomer_enumerator.Canonicalize(mol_uncharged)

            refined_smiles = Chem.MolToSmiles(canonical_mol, isomericSmiles=True)
            refined_inchikey = Chem.MolToInchiKey(canonical_mol)

            # Track if a proton shift actually altered the structural string representation
            if refined_smiles != profiled_smiles:
                tautomers_shifted_count += 1

        except Exception:
            # Safe structural fallback
            refined_smiles = profiled_smiles
            refined_inchikey = old_inchikey

        tautomer_records.append({
            'Compound_ID': comp_id,
            'Profiled_SMILES': profiled_smiles,
            'Refined_SMILES': refined_smiles,
            'Original_InChIKey': old_inchikey,
            'Refined_InChIKey': refined_inchikey
        })

    # Build the final evaluation matrix
    df_tautomer_report = pd.DataFrame(tautomer_records)

    # Save a permanent structural transition record to the workspace
    df_tautomer_report.to_csv("Plitidepsin_Analogs_Tautomer_Refinement_History.csv", index=False)

    # Save out the definitive ready-to-dock 2D master base file
    df_final_2d_master = df_tautomer_report[['Compound_ID', 'Refined_SMILES', 'Refined_InChIKey']].copy()
    df_final_2d_master.columns = ['Compound_ID', 'SMILES', 'InChIKey_Hash']
    df_final_2d_master.to_csv("Plitidepsin_Analogs_Final_2D_Master.csv", index=False)

    print("✨ Physiological Tautomerization Subroutine Complete!")
    print(f"🟩 Total structures evaluated and locked at pH 7.4: {len(df_tautomer_report)}")
    print(f"🔄 Shifted/optimized proton tautomers: {tautomers_shifted_count}")
    print(f"📁 Transformation report log saved as: 'Plitidepsin_Analogs_Tautomer_Refinement_History.csv'")
    print(f"📁 DEFINITIVE 2D master collection saved as: 'Plitidepsin_Analogs_Final_2D_Master.csv'")

    return df_tautomer_report

# --- Run Tautomer Standardization Framework ---
df_tautomer_report = run_production_physiological_tautomerization()

# View interactive tracking logs
df_tautomer_report

🧬 Loading screened macrocyclic library: 'Plitidepsin_Analogs_Profiled_Master_2D.csv'...
⚡ Processing 108 molecules through physiological equilibrium loops...



[00:49:10] Running Uncharger
[00:49:12] Tautomer enumeration stopped at 688 tautomers: max transforms reached
[00:49:12] Running Uncharger
[00:49:14] Tautomer enumeration stopped at 688 tautomers: max transforms reached
[00:49:14] Running Uncharger
[00:49:15] Tautomer enumeration stopped at 568 tautomers: max transforms reached
[00:49:15] Running Uncharger
[00:49:17] Running Uncharger
[00:49:18] Tautomer enumeration stopped at 688 tautomers: max transforms reached
[00:49:18] Running Uncharger
[00:49:19] Tautomer enumeration stopped at 688 tautomers: max transforms reached
[00:49:19] Running Uncharger
[00:49:20] Tautomer enumeration stopped at 568 tautomers: max transforms reached
[00:49:20] Running Uncharger
[00:49:22] Tautomer enumeration stopped at 520 tautomers: max transforms reached
[00:49:22] Running Uncharger
[00:49:23] Tautomer enumeration stopped at 724 tautomers: max transforms reached
[00:49:23] Running Uncharger
[00:49:25] Tautomer enumeration stopped at 724 tautomers: max 

✨ Physiological Tautomerization Subroutine Complete!
🟩 Total structures evaluated and locked at pH 7.4: 108
🔄 Shifted/optimized proton tautomers: 100
📁 Transformation report log saved as: 'Plitidepsin_Analogs_Tautomer_Refinement_History.csv'
📁 DEFINITIVE 2D master collection saved as: 'Plitidepsin_Analogs_Final_2D_Master.csv'


[00:51:35] Tautomer enumeration stopped at 570 tautomers: max transforms reached


,Compound_ID,Profiled_SMILES,Refined_SMILES,Original_InChIKey,Refined_InChIKey
0,CID_123844,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)C(NC(=O)C(CC(C)C)NC)[C@@...,XQZOGOCTPKFYKC-VSZULPIASA-N,XQZOGOCTPKFYKC-BHORLZIBSA-N
1,CID_44355259,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)C(NC(=O)C(CC(C)C)NC)[C@@...,XQZOGOCTPKFYKC-WWBUNASKSA-N,XQZOGOCTPKFYKC-BHORLZIBSA-N
2,CID_44592548,CC[C@H](C)[C@@H]1NC(=O)[C@@H](N2CN(C)[C@H](CC(...,CC[C@H](C)[C@@H]1NC(=O)C(N2CN(C)C(CC(C)C)C2=O)...,LSLAZLSLSCFXNW-MGCJBGHFSA-N,LSLAZLSLSCFXNW-JZXXXIDQSA-N
3,CID_44592549,CC[C@H](C)[C@@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(...,CC[C@H](C)[C@@H]1NC(=O)C(NC(=O)C(CC(C)C)N(C)C(...,TXZGIJLIQOCJFR-NXEGRYKESA-N,TXZGIJLIQOCJFR-WKXFPTBMSA-N
4,CID_44592553,CC[C@H](C)[C@@H]1NC(=O)[C@@H](NC(=O)[C@H](CC(C...,CC[C@H](C)[C@@H]1NC(=O)C(NC(=O)C(CC(C)C)N(C)C=...,LUVOBHRWQNQKGY-GVVJTJSGSA-N,LUVOBHRWQNQKGY-JZXXXIDQSA-N
...,...,...,...,...,...
103,CID_177436729,CC[C@H](C)C1NC(=O)C(NC(=O)[C@@H](CC(C)C)N(C)C=...,CC[C@H](C)C1NC(=O)C(NC(=O)C(CC(C)C)N(C)C=O)[C@...,LUVOBHRWQNQKGY-CGKXRRSUSA-N,LUVOBHRWQNQKGY-MTTJRWLGSA-N
104,CID_177467781,CC[C@H](C)[C@H]1NC(=O)[C@@H](NC(=O)[C@@H](CC(C...,CC[C@H](C)[C@H]1NC(=O)C(NC(=O)C(CC(C)C)N(C)C(=...,WFGVMVUFPQNINY-OKIXPNRASA-N,WFGVMVUFPQNINY-QCSRQKMTSA-N
105,CID_177536164,CC[C@H](C)C1NC(=O)C(N2CN(C)[C@@H](CC(C)C)C2=O)...,CC[C@H](C)C1NC(=O)C(N2CN(C)C(CC(C)C)C2=O)[C@@H...,LSLAZLSLSCFXNW-KFVVOLBJSA-N,LSLAZLSLSCFXNW-MTTJRWLGSA-N
106,CID_177567562,CN[C@H](CC(C)C)C(=O)N[C@@H]1C(=O)N[C@@H](C(C)C...,CNC(CC(C)C)C(=O)NC1C(=O)N[C@@H](C(C)C)[C@@H](O...,DMQOHQUDFDJQGG-NBFKFKEHSA-N,DMQOHQUDFDJQGG-KRXHTSEVSA-N


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from google.colab import data_table

# Maintain interactive tracking grids in Colab
data_table.enable_dataframe_formatter()

def run_production_stereochemical_enumeration(input_csv="Plitidepsin_Analogs_Final_2D_Master.csv"):
    """
    Ingests the 108 physiologically refined macrocycles, audits them for ambiguous
    chiral centers, and expands unassigned structures into distinct, trackable stereoisomers.
    """
    print(f"🧬 Ingesting physiological 2D master library: '{input_csv}'...")
    try:
        df_final_2d = pd.read_csv(input_csv)
    except Exception:
        print(f"❌ Error: Could not locate '{input_csv}'. Verify Step 7 finished completely.")
        return None

    # Configure the enumeration options to ONLY expand unassigned centers
    # This protects all explicitly declared chiral bonds from your source data
    options = StereoEnumerationOptions(tryEmbedding=True, onlyUnassigned=True)

    expanded_rows = []
    compounds_with_chiral_gaps = 0
    total_isomers_generated = 0

    print(f"⚡ Auditing stereocenters across {len(df_final_2d)} validated leads...\n")

    for idx, row in df_final_2d.iterrows():
        comp_id = row['Compound_ID']
        parent_smiles = row['SMILES']

        mol = Chem.MolFromSmiles(parent_smiles)
        if mol is None:
            continue

        # Execute targeted chiral enumeration
        isomers = list(EnumerateStereoisomers(mol, options=options))
        num_isomers = len(isomers)

        # If more than 1 isomer is generated, an unassigned center was detected and resolved
        if num_isomers > 1:
            compounds_with_chiral_gaps += 1

        for i, isomer_mol in enumerate(isomers):
            total_isomers_generated += 1
            isomer_smiles = Chem.MolToSmiles(isomer_mol, isomericSmiles=True)
            isomer_inchikey = Chem.MolToInchiKey(isomer_mol)

            # Create a unique tracking ID for each structural isomer variant
            # If a compound has zero unassigned centers, it yields exactly 1 variant (iso_1)
            variant_id = f"{comp_id}_iso_{i+1}"

            expanded_rows.append({
                'Parent_Compound_ID': comp_id,
                'Isomer_Variant_ID': variant_id,
                'Isomer_SMILES': isomer_smiles,
                'Isomer_InChIKey': isomer_inchikey,
                'Stereo_Expansion': 'Fully Specified' if num_isomers == 1 else f'Enumerated ({i+1}/{num_isomers})'
            })

    # Compile the final stereochemically validated database
    df_stereo_master = pd.DataFrame(expanded_rows)

    # Save a permanent copy of the 2D stereochemical master file to disk workspace
    df_stereo_master.to_csv("Plitidepsin_Analogs_Stereo_Master_2D.csv", index=False)

    print("✨ Stereochemical Audit and Expansion Complete!")
    print(f"🟩 Total parent compounds evaluated: {len(df_final_2d)}")
    print(f"⚠️ Parent compounds with unassigned/flat stereocenters: {compounds_with_chiral_gaps}")
    print(f"🎯 Total fully specified isomer leads generated: {total_isomers_generated}")
    print(f"📁 Definitive 2D entry pool saved as: 'Plitidepsin_Analogs_Stereo_Master_2D.csv'")

    return df_stereo_master

# --- Execute Stereochemical Validation Loop ---
df_stereo_master = run_production_stereochemical_enumeration()

# View the final stereochemically validated library
df_stereo_master

🧬 Ingesting physiological 2D master library: 'Plitidepsin_Analogs_Final_2D_Master.csv'...
⚡ Auditing stereocenters across 108 validated leads...

